# BUS4 110A — Data Visualization Demo

### Learning goals
By the end of the demo, students should be able to:

- explain why data visualization is useful for business analysis
- prepare summarized data for visualization using Pandas
- use `groupby()` and `reset_index()` to prepare data for plotting
- choose an appropriate chart based on the business question
- create and interpret **bar, pie, line, and scatter** charts

> The focus is not memorizing every plotting option.  
> The focus is translating a **business question → data preparation → chart choice → interpretation**.


## Part 1 — Setup

We will continue using the same `TableauSalesData.xlsx` dataset from the Pandas notebooks.


In [2]:
import pandas as pd
sales = pd.read_csv("TableauSalesData.csv")
print(sales.head())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0    4919  CA-2017-160304  2017-01-02  2017-01-07  Standard Class    BM-11575   
1    4920  CA-2017-160304  2017-01-02  2017-01-07  Standard Class    BM-11575   
2    3074  CA-2017-125206  2017-01-03  2017-01-05     First Class    LR-16915   
3    8604  US-2017-116365  2017-01-03  2017-01-08  Standard Class    CA-12310   
4    8605  US-2017-116365  2017-01-03  2017-01-08  Standard Class    CA-12310   

       Customer Name    Segment        Country          City  ... Postal Code  \
0      Brendan Murry  Corporate  United States  Gaithersburg  ...       20877   
1      Brendan Murry  Corporate  United States  Gaithersburg  ...       20877   
2       Lena Radford   Consumer  United States   Los Angeles  ...       90045   
3  Christine Abelman  Corporate  United States   San Antonio  ...       78207   
4  Christine Abelman  Corporate  United States   San Antonio  ...       78207   

    Region       Product I

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Change this path if your file is stored somewhere else in Google Drive
sales = pd.read_csv("TableauSalesData.csv")

sales.head()


## Part 2 — Prepare data for visualization

A chart usually needs **summarized data**, not every transaction.

Business question:

> How did total Sales change by year?


In [ ]:
# Make sure Order Date is stored as datetime
sales["Order Date"] = pd.to_datetime(sales["Order Date"])

# Extract year
sales["Year"] = sales["Order Date"].dt.year

# Group by year and calculate total sales
yearly_sales = sales.groupby("Year")[["Sales"]].sum()

yearly_sales


In [ ]:
yearly_sales_plot = yearly_sales.reset_index()
yearly_sales_plot


# Part 3 — Bar Chart
### Best for: **Comparison**

Business question:

> Which region has the highest total sales?


In [ ]:
region_sales = (
    sales.groupby("Region")[["Sales"]]
         .sum()
         .reset_index()
         .sort_values("Sales", ascending=False)
)

region_sales


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=region_sales,
    x="Region",
    y="Sales"
)

plt.title("Total Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales ($)")
plt.show()


### Questions to check

1. What is being compared?
2. Which region has the highest total Sales?


# Part 4 — Pie Chart
### Best for: **Composition**

Business question:

> What share of total sales comes from each customer segment?

Pie charts work best when:
- there are only a few categories
- the parts add up to a meaningful whole
- differences are large enough to compare


In [ ]:
segment_sales = (
    sales.groupby("Segment")[["Sales"]]
         .sum()
         .reset_index()
)

segment_sales


In [ ]:
plt.figure(figsize=(6, 6))

plt.pie(
    segment_sales["Sales"],
    labels=segment_sales["Segment"],
    autopct="%1.1f%%"
)

plt.title("Share of Sales by Customer Segment")
plt.show()


### Discussion

A pie chart can show composition, but exact comparisons are usually easier with a bar chart.

Let's display the **same data** as a bar chart.


In [ ]:
plt.figure(figsize=(7, 4))

sns.barplot(
    data=segment_sales.sort_values("Sales", ascending=False),
    x="Segment",
    y="Sales"
)

plt.title("Sales by Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Sales ($)")
plt.show()


### Question to check

Which chart makes it easier to compare small differences: the pie chart or the bar chart?


# Part 5 — Line Chart
### Best for: **Evolution over time**

Business question:

> How have Table sales changed over time?


In [ ]:
tables = sales.loc[sales["Sub-Category"] == "Tables"].copy()

table_year_sales = (
    tables.groupby("Year")[["Sales"]]
          .sum()
          .reset_index()
)

table_year_sales


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    table_year_sales["Year"],
    table_year_sales["Sales"],
    marker="o"
)

plt.title("Table Sales by Year")
plt.xlabel("Year")
plt.ylabel("Sales ($)")
plt.xticks(table_year_sales["Year"])
plt.show()


###

Why does a line chart make sense here?

Because the x-axis represents an **ordered time sequence**, and we care about the pattern of change.


# Part 6 — Scatterplot
### Best for: **Relationship**

Business question:

> Is Discount related to Profit?


In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=sales,
    x="Discount",
    y="Profit",
    alpha=0.5
)

plt.title("Relationship Between Discount and Profit")
plt.xlabel("Discount")
plt.ylabel("Profit ($)")
plt.show()


### Connect to correlation

A scatterplot helps us **see** a relationship.

A correlation coefficient helps us **measure** the direction and strength of that relationship.

Remember:

- correlation is between `-1` and `1`
- positive → variables tend to move together
- negative → one tends to decrease as the other increases
- correlation does **not** prove causation


In [ ]:
sales[["Discount", "Profit"]].corr()


# Part 7 — Choosing the Right Chart

A useful rule of thumb:

| Business Question | Chart |
|---|---|
| Compare categories | **Bar** |
| Show composition / share | **Pie** |
| Show change over time | **Line** |
| Examine relationship between two numeric variables | **Scatterplot** |

The chart should be chosen based on the **question**, not because a chart looks interesting.
